# Business Problem
Artea is an online retailer specialized in selling handmade items. Despite high metric performance on time spent on website and referral rates, only 13% of the website visitor made transactions. To help incentivize purchases, Artea aims to send discounted coupons to users who are registered but had not made a transaction in the last 2 months. Prior to sending coupons, Artea’s data science team needs to run A/B test to experiment if sending discounted coupons can increase sales. 

# Business Goal
The goal of this project is to examine data from the A/B test to examine causal and monetization effect of Artea’s discounted coupons campaign on sales. 



### Install Packages

In [1]:
import pandas as pd
import scipy.stats as stats
from scipy.stats import ttest_ind
from scipy.stats import chi2_contingency
import statsmodels.formula.api as smf
from statsmodels.formula.api import ols, logit
from sklearn.preprocessing import OneHotEncoder
import matplotlib.pyplot as plt

ImportError: cannot import name 'ComplexWarning' from 'numpy.core.numeric' (/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/numpy/core/numeric.py)

### Upload Data

In [3]:
PATH = '/Users/chohasong/Desktop/Python Analytics/Fall_2024/B9651_Marketing/'
ab_test = pd.read_excel(PATH+'artea.xlsx',sheet_name='AB_test')

In [4]:
ab_test.head(5)

,id,trans_after,revenue_after,test_coupon,channel_acq,num_past_purch,spent_last_purchase,weeks_since_visit,browsing_minutes,shopping_cart
0,AB_1,0,0.0,0,2,6,62.99,6,1,0
1,AB_2,0,0.0,1,1,2,53.99,0,7,1
2,AB_3,0,0.0,1,2,3,88.98,3,4,0
3,AB_4,0,0.0,0,2,1,68.99,1,19,0
4,AB_5,0,0.0,1,3,3,66.49,4,20,0


In [5]:
ab_test.shape

(5000, 10)

In [6]:
ab_test['trans_after'].value_counts()

trans_after
0    4423
1     480
2      80
3      15
5       1
4       1
Name: count, dtype: int64

# Randomization Controls 
There are multiple methods to test whether the team successfully randomized treatment versus non treatment groups.

Method A: Logistic Regression 
- Logistic Regression is of coupon assignment on all user variables. 
- Null Hypothesis: spending on last purchase coefficient is 0 (the variable has no effect on predicting the likelihood of receiving a test coupon)
- Alternative Hypothesis: spending on last purchase coefficient is not 0.

Method B: T-test 
- T-test is on continuous user variables ( number of past purchases, spending on last purchase, weeks since visit, and browsing minutes ) 
- Hypothesis: sample mean of number of past purchases in coupon group is equal to sample mean of number of past purchases in non coupon group
- Null Hypothesis: sample mean of number of past purchases in coupon group is different from sample mean of number of past purchases in non coupon group

Method C: Chi-squared Test 
- Chi- squared test is on categorical user variables ( shopping cart, channel acquisition) 
- Null Hypothesis: Shopping cart and coupon group have no association.
- Alternative Hypothesis: Shopping cart and coupon group have association.



In [58]:
# see number of customers who received discounted coupons vs who did not
ab_test['test_coupon'].value_counts()

test_coupon
1    2502
0    2498
Name: count, dtype: int64

In [59]:
# form variables based on coupon received status
treatment = ab_test[ab_test['test_coupon']==1]
control = ab_test[ab_test['test_coupon']==0]

In [60]:
# Method A: test if test coupon distribution is predictable using all variables but revenue and transactions 
log_model = logit('test_coupon ~ num_past_purch + spent_last_purchase + weeks_since_visit + browsing_minutes + shopping_cart + channel_acq',data = ab_test).fit()
log_model.summary()

Optimization terminated successfully.
         Current function value: 0.692735
         Iterations 3


<class 'statsmodels.iolib.summary.Summary'>
"""
                           Logit Regression Results                           
==============================================================================
Dep. Variable:            test_coupon   No. Observations:                 5000
Model:                          Logit   Df Residuals:                     4993
Method:                           MLE   Df Model:                            6
Date:                Sat, 21 Dec 2024   Pseudo R-squ.:               0.0005937
Time:                        16:22:59   Log-Likelihood:                -3463.7
converged:                       True   LL-Null:                       -3465.7
Covariance Type:            nonrobust   LLR p-value:                    0.6611
=======================================================================================
                          coef    std err          z      P>|z|      [0.025      0.975]
---------------------------------------------------------------------------------------
Intercept              -0.0579      0.101     -0.574      0.566      -0.256       0.140
num_past_purch          0.0080      0.012      0.658      0.510      -0.016       0.032
spent_last_purchase     0.0003      0.001      0.586      0.558      -0.001       0.001
weeks_since_visit       0.0147      0.013      1.174      0.240      -0.010       0.039
browsing_minutes       -0.0015      0.004     -0.360      0.719      -0.010       0.007
shopping_cart          -0.0713      0.063     -1.133      0.257      -0.195       0.052
channel_acq             0.0087      0.027      0.322      0.747      -0.044       0.061
=======================================================================================
"""

In [61]:
# Method B: test if there is statistical difference in sample means between continuous variables of coupon vs non coupon group

# select continuous variables to conduct t test
variables_to_test = ['num_past_purch', 'spent_last_purchase', 'weeks_since_visit', 'browsing_minutes']

for variable in variables_to_test:
    t_stat, p_value = ttest_ind(control[variable], treatment[variable], equal_var=True,alternative='two-sided')
    print(f"{variable}: t-statistic={t_stat:.2f}, p-value={p_value:.3f}")

num_past_purch: t-statistic=-0.99, p-value=0.321
spent_last_purchase: t-statistic=-0.93, p-value=0.350
weeks_since_visit: t-statistic=-1.16, p-value=0.246
browsing_minutes: t-statistic=0.19, p-value=0.852


In [62]:
# Method C: test independence of two selected variables 
crosstab = pd.crosstab(ab_test['shopping_cart'], ab_test['test_coupon'])
chi2, p, dof, expected = chi2_contingency(crosstab)
print(f"Chi2 Stat: {chi2:.4f}")
print(f"P-value: {p:.4f}")
print(f"Degrees of Freedom: {dof}")
print(f"Expected Frequencies:\n{expected}")

Chi2 Stat: 1.1278
P-value: 0.2882
Degrees of Freedom: 1
Expected Frequencies:
[[1766.5856 1769.4144]
 [ 731.4144  732.5856]]


In [63]:
# Method C: test independence of two selected variables 
crosstab = pd.crosstab(ab_test['channel_acq'], ab_test['test_coupon'])
chi2, p, dof, expected = chi2_contingency(crosstab)
print(f"Chi2 Stat: {chi2:.4f}")
print(f"P-value: {p:.4f}")
print(f"Degrees of Freedom: {dof}")
print(f"Expected Frequencies:\n{expected}")

Chi2 Stat: 1.7821
P-value: 0.7757
Degrees of Freedom: 4
Expected Frequencies:
[[1016.686  1018.314 ]
 [ 529.0764  529.9236]
 [ 775.3792  776.6208]
 [ 118.9048  119.0952]
 [  57.9536   58.0464]]


All of the p values are above 0.05, so we reject null hypothesis and conclude that there was proper randomization control.

# Causality Check
Did the coupon increase revenues? 


In [64]:
treatment['revenue_after'].mean()

7.53867306155076

In [65]:
control['revenue_after'].mean()

7.780168134507607

In [66]:
lr = smf.ols('revenue_after~test_coupon', data = ab_test)
lr_result = lr.fit()

In [67]:
lr_result.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:          revenue_after   R-squared:                       0.000
Model:                            OLS   Adj. R-squared:                 -0.000
Method:                 Least Squares   F-statistic:                    0.1306
Date:                Sat, 21 Dec 2024   Prob (F-statistic):              0.718
Time:                        16:22:59   Log-Likelihood:                -22906.
No. Observations:                5000   AIC:                         4.582e+04
Df Residuals:                    4998   BIC:                         4.583e+04
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
===============================================================================
                  coef    std err          t      P>|t|      [0.025      0.975]
-------------------------------------------------------------------------------
Intercept       7.7802      0.473     16.456      0.000       6.853       8.707
test_coupon    -0.2415      0.668     -0.361      0.718      -1.552       1.069
==============================================================================
Omnibus:                     3946.015   Durbin-Watson:                   1.968
Prob(Omnibus):                  0.000   Jarque-Bera (JB):            74926.257
Skew:                           3.764   Prob(JB):                         0.00
Kurtosis:                      20.406   Cond. No.                         2.62
==============================================================================

Notes:
[1] Standard Errors assume that the covariance matrix of the errors is correctly specified.
"""

Did it increase transactions?

In [68]:
control['trans_after'].mean()

0.1257005604483587

In [69]:
treatment['trans_after'].mean()

0.1518784972022382

In [70]:
lr_transaction = smf.ols('trans_after~test_coupon', data = ab_test)
lr_transaction_result = lr_transaction.fit()
lr_transaction_result.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:            trans_after   R-squared:                       0.001
Model:                            OLS   Adj. R-squared:                  0.001
Method:                 Least Squares   F-statistic:                     4.872
Date:                Sat, 21 Dec 2024   Prob (F-statistic):             0.0273
Time:                        16:22:59   Log-Likelihood:                -2748.1
No. Observations:                5000   AIC:                             5500.
Df Residuals:                    4998   BIC:                             5513.
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
===============================================================================
                  coef    std err          t      P>|t|      [0.025      0.975]
-------------------------------------------------------------------------------
Intercept       0.1257      0.008     14.982      0.000       0.109       0.142
test_coupon     0.0262      0.012      2.207      0.027       0.003       0.049
==============================================================================
Omnibus:                     3814.872   Durbin-Watson:                   1.963
Prob(Omnibus):                  0.000   Jarque-Bera (JB):            66979.526
Skew:                           3.610   Prob(JB):                         0.00
Kurtosis:                      19.413   Cond. No.                         2.62
==============================================================================

Notes:
[1] Standard Errors assume that the covariance matrix of the errors is correctly specified.
"""

# Q3
Which customers should be targeted in the next targeting campaign?

In [71]:
ab_test_dummy=pd.get_dummies(ab_test,columns=['channel_acq'], drop_first=True)

In [72]:
ab_test_dummy.columns

Index(['id', 'trans_after', 'revenue_after', 'test_coupon', 'num_past_purch',
       'spent_last_purchase', 'weeks_since_visit', 'browsing_minutes',
       'shopping_cart', 'channel_acq_2', 'channel_acq_3', 'channel_acq_4',
       'channel_acq_5'],
      dtype='object')

In [73]:
ab_test['channel_acq'] = ab_test['channel_acq'].astype('category')


In [74]:
# channel acq cart
hte_channel = smf.ols('revenue_after~test_coupon+C(channel_acq)+test_coupon*C(channel_acq)', data = ab_test)
hte_result_channel = hte_channel.fit()
hte_result_channel.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:          revenue_after   R-squared:                       0.020
Model:                            OLS   Adj. R-squared:                  0.018
Method:                 Least Squares   F-statistic:                     11.07
Date:                Sat, 21 Dec 2024   Prob (F-statistic):           2.88e-17
Time:                        16:22:59   Log-Likelihood:                -22857.
No. Observations:                5000   AIC:                         4.573e+04
Df Residuals:                    4990   BIC:                         4.580e+04
Df Model:                           9                                         
Covariance Type:            nonrobust                                         
===================================================================================================
                                      coef    std err          t      P>|t|      [0.025      0.975]
---------------------------------------------------------------------------------------------------
Intercept                           4.9223      0.731      6.737      0.000       3.490       6.355
C(channel_acq)[T.2]                 4.8926      1.264      3.870      0.000       2.414       7.371
C(channel_acq)[T.3]                 4.2495      1.110      3.828      0.000       2.073       6.426
C(channel_acq)[T.4]                 6.2573      2.321      2.696      0.007       1.707      10.807
C(channel_acq)[T.5]                 9.9372      3.160      3.144      0.002       3.742      16.133
test_coupon                        -2.1184      1.038     -2.040      0.041      -4.154      -0.083
test_coupon:C(channel_acq)[T.2]     3.1988      1.775      1.802      0.072      -0.281       6.678
test_coupon:C(channel_acq)[T.3]     3.2840      1.578      2.081      0.038       0.190       6.378
test_coupon:C(channel_acq)[T.4]     2.4569      3.212      0.765      0.444      -3.840       8.754
test_coupon:C(channel_acq)[T.5]     0.0156      4.470      0.003      0.997      -8.749       8.780
==============================================================================
Omnibus:                     3899.658   Durbin-Watson:                   1.973
Prob(Omnibus):                  0.000   Jarque-Bera (JB):            73330.908
Skew:                           3.701   Prob(JB):                         0.00
Kurtosis:                      20.240   Cond. No.                         19.2
==============================================================================

Notes:
[1] Standard Errors assume that the covariance matrix of the errors is correctly specified.
"""

In [75]:
ab_test['shopping_cart'].value_counts()

shopping_cart
0    3536
1    1464
Name: count, dtype: int64

In [76]:
# shopping cart
hte_shopping = smf.ols('revenue_after~test_coupon+shopping_cart+test_coupon*shopping_cart', data = ab_test)
hte_result_shopping = hte_shopping.fit()
hte_result_shopping.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:          revenue_after   R-squared:                       0.029
Model:                            OLS   Adj. R-squared:                  0.028
Method:                 Least Squares   F-statistic:                     49.02
Date:                Sat, 21 Dec 2024   Prob (F-statistic):           3.25e-31
Time:                        16:22:59   Log-Likelihood:                -22834.
No. Observations:                5000   AIC:                         4.568e+04
Df Residuals:                    4996   BIC:                         4.570e+04
Df Model:                           3                                         
Covariance Type:            nonrobust                                         
=============================================================================================
                                coef    std err          t      P>|t|      [0.025      0.975]
---------------------------------------------------------------------------------------------
Intercept                     5.4817      0.557      9.841      0.000       4.390       6.574
test_coupon                  -0.7395      0.784     -0.944      0.345      -2.276       0.797
shopping_cart                 7.6657      1.017      7.536      0.000       5.672       9.660
test_coupon:shopping_cart     2.1202      1.448      1.464      0.143      -0.719       4.959
==============================================================================
Omnibus:                     3868.063   Durbin-Watson:                   1.972
Prob(Omnibus):                  0.000   Jarque-Bera (JB):            71758.667
Skew:                           3.661   Prob(JB):                         0.00
Kurtosis:                      20.054   Cond. No.                         6.34
==============================================================================

Notes:
[1] Standard Errors assume that the covariance matrix of the errors is correctly specified.
"""

In [77]:
ab_test.columns

Index(['id', 'trans_after', 'revenue_after', 'test_coupon', 'channel_acq',
       'num_past_purch', 'spent_last_purchase', 'weeks_since_visit',
       'browsing_minutes', 'shopping_cart'],
      dtype='object')

In [78]:
# shopping cart
hte_num_past_purch = smf.ols('revenue_after~test_coupon+num_past_purch+test_coupon*num_past_purch', data = ab_test)
hte_result_num_past_purch = hte_num_past_purch.fit()
hte_result_num_past_purch.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:          revenue_after   R-squared:                       0.115
Model:                            OLS   Adj. R-squared:                  0.114
Method:                 Least Squares   F-statistic:                     215.5
Date:                Sat, 21 Dec 2024   Prob (F-statistic):          1.94e-131
Time:                        16:23:00   Log-Likelihood:                -22602.
No. Observations:                5000   AIC:                         4.521e+04
Df Residuals:                    4996   BIC:                         4.524e+04
Df Model:                           3                                         
Covariance Type:            nonrobust                                         
==============================================================================================
                                 coef    std err          t      P>|t|      [0.025      0.975]
----------------------------------------------------------------------------------------------
Intercept                      0.3259      0.573      0.569      0.569      -0.797       1.449
test_coupon                    1.9734      0.807      2.446      0.014       0.391       3.555
num_past_purch                 3.6909      0.179     20.674      0.000       3.341       4.041
test_coupon:num_past_purch    -1.1859      0.246     -4.819      0.000      -1.668      -0.703
==============================================================================
Omnibus:                     3506.441   Durbin-Watson:                   1.947
Prob(Omnibus):                  0.000   Jarque-Bera (JB):            54801.055
Skew:                           3.230   Prob(JB):                         0.00
Kurtosis:                      17.876   Cond. No.                         11.6
==============================================================================

Notes:
[1] Standard Errors assume that the covariance matrix of the errors is correctly specified.
"""

In [79]:
ab_test.columns

Index(['id', 'trans_after', 'revenue_after', 'test_coupon', 'channel_acq',
       'num_past_purch', 'spent_last_purchase', 'weeks_since_visit',
       'browsing_minutes', 'shopping_cart'],
      dtype='object')

In [80]:
hte = smf.ols('revenue_after~test_coupon+C(channel_acq)+test_coupon*C(channel_acq)+num_past_purch+test_coupon*num_past_purch', data = ab_test)
model = hte.fit()
model.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:          revenue_after   R-squared:                       0.133
Model:                            OLS   Adj. R-squared:                  0.131
Method:                 Least Squares   F-statistic:                     69.35
Date:                Sat, 21 Dec 2024   Prob (F-statistic):          3.00e-145
Time:                        16:23:00   Log-Likelihood:                -22550.
No. Observations:                5000   AIC:                         4.512e+04
Df Residuals:                    4988   BIC:                         4.520e+04
Df Model:                          11                                         
Covariance Type:            nonrobust                                         
===================================================================================================
                                      coef    std err          t      P>|t|      [0.025      0.975]
---------------------------------------------------------------------------------------------------
Intercept                          -2.5195      0.775     -3.253      0.001      -4.038      -1.001
C(channel_acq)[T.2]                 5.0504      1.189      4.246      0.000       2.719       7.382
C(channel_acq)[T.3]                 4.2625      1.044      4.081      0.000       2.215       6.310
C(channel_acq)[T.4]                 5.3753      2.184      2.462      0.014       1.094       9.656
C(channel_acq)[T.5]                 9.7475      2.973      3.279      0.001       3.919      15.576
test_coupon                         0.3825      1.093      0.350      0.726      -1.760       2.525
test_coupon:C(channel_acq)[T.2]     2.5051      1.670      1.500      0.134      -0.769       5.779
test_coupon:C(channel_acq)[T.3]     3.1547      1.485      2.125      0.034       0.244       6.066
test_coupon:C(channel_acq)[T.4]     2.6019      3.022      0.861      0.389      -3.323       8.527
test_coupon:C(channel_acq)[T.5]    -0.8447      4.206     -0.201      0.841      -9.091       7.401
num_past_purch                      3.6886      0.177     20.854      0.000       3.342       4.035
test_coupon:num_past_purch         -1.2242      0.244     -5.020      0.000      -1.702      -0.746
==============================================================================
Omnibus:                     3438.657   Durbin-Watson:                   1.955
Prob(Omnibus):                  0.000   Jarque-Bera (JB):            52588.711
Skew:                           3.148   Prob(JB):                         0.00
Kurtosis:                      17.587   Cond. No.                         60.5
==============================================================================

Notes:
[1] Standard Errors assume that the covariance matrix of the errors is correctly specified.
"""

In [81]:
# shopping cart
hte_browsing_minutes = smf.ols('revenue_after~ C(channel_acq)+test_coupon*C(channel_acq)+ test_coupon+shopping_cart+test_coupon*shopping_cart', data = ab_test)
hte_result_browsing_minutes = hte_browsing_minutes.fit()
hte_result_browsing_minutes.summary()
hte_result_browsing_minutes.predict()

array([ 7.52253256,  9.80741981,  7.61017801, ..., -0.06607833,
        9.80741981,  2.41005241])

In [82]:
ab_test.columns

Index(['id', 'trans_after', 'revenue_after', 'test_coupon', 'channel_acq',
       'num_past_purch', 'spent_last_purchase', 'weeks_since_visit',
       'browsing_minutes', 'shopping_cart'],
      dtype='object')

In [83]:
ab_test.groupby('channel_acq')['revenue_after'].mean()

channel_acq
1     3.872982
2    10.369896
3     9.747805
4    11.357353
5    13.808086
Name: revenue_after, dtype: float64

# Q4


In [84]:
next_campaign = pd.read_excel(PATH+'artea.xlsx',sheet_name='Next_Campaign')

In [85]:
next_campaign

,id,channel_acq,num_past_purch,spent_last_purchase,weeks_since_visit,browsing_minutes,shopping_cart
0,Next_1,1,0,0.00,0,16,1
1,Next_2,3,0,0.00,2,19,0
2,Next_3,4,5,63.99,1,14,0
3,Next_4,1,0,0.00,7,15,0
4,Next_5,2,2,52.49,2,19,0
...,...,...,...,...,...,...,...
5995,Next_5996,1,0,0.00,4,2,1
5996,Next_5997,1,0,0.00,5,13,0
5997,Next_5998,2,2,59.99,1,1,1
5998,Next_5999,2,1,113.98,6,22,0


In [86]:
ab_test_dummy.columns

Index(['id', 'trans_after', 'revenue_after', 'test_coupon', 'num_past_purch',
       'spent_last_purchase', 'weeks_since_visit', 'browsing_minutes',
       'shopping_cart', 'channel_acq_2', 'channel_acq_3', 'channel_acq_4',
       'channel_acq_5'],
      dtype='object')

In [87]:
ab_test.loc[(ab_test['channel_acq']==2 ) | (ab_test['channel_acq']==3),'test_coupon'] = 1

In [88]:
next_campaign.head(5)

,id,channel_acq,num_past_purch,spent_last_purchase,weeks_since_visit,browsing_minutes,shopping_cart
0,Next_1,1,0,0.00,0,16,1
1,Next_2,3,0,0.00,2,19,0
2,Next_3,4,5,63.99,1,14,0
3,Next_4,1,0,0.00,7,15,0
4,Next_5,2,2,52.49,2,19,0


In [89]:
next_campaign['test_coupon'] = 0

In [90]:
next_campaign.loc[(next_campaign['channel_acq'] == 2 ) | (next_campaign['channel_acq'] == 3 ),'test_coupon'] = 1

In [91]:
next_campaign.loc[(next_campaign['num_past_purch'] == 0 ),'test_coupon'] = 1

In [92]:
# shopping cart
model_pred = smf.ols('revenue_after~ test_coupon+ C(channel_acq)+test_coupon*C(channel_acq)+ num_past_purch + num_past_purch*test_coupon ', data = ab_test)
model_pred_fit = model_pred.fit()
model_pred_fit.summary()
next_campaign['revenue_after'] = model_pred_fit.predict(next_campaign)

In [93]:
next_campaign

,id,channel_acq,num_past_purch,spent_last_purchase,weeks_since_visit,browsing_minutes,shopping_cart,test_coupon,revenue_after
0,Next_1,1,0,0.00,0,16,1,1,-2.997151
1,Next_2,3,0,0.00,2,19,0,1,3.863892
2,Next_3,4,5,63.99,1,14,0,0,20.779308
3,Next_4,1,0,0.00,7,15,0,1,-2.997151
4,Next_5,2,2,52.49,2,19,0,1,10.002441
...,...,...,...,...,...,...,...,...,...
5995,Next_5996,1,0,0.00,4,2,1,1,-2.997151
5996,Next_5997,1,0,0.00,5,13,0,1,-2.997151
5997,Next_5998,2,2,59.99,1,1,1,1,10.002441
5998,Next_5999,2,1,113.98,6,22,0,1,7.099121


In [94]:
next_campaign['revenue_after'].sum()

47319.006743009195